In [26]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import os

# Load dataset
df = pd.read_pickle('processed_data/music_dataset_10000_songs.pkl')


In [27]:
# Prepare features for recommendation
# Select numeric features (exclude non-numeric columns)
numeric_features = [
    'artist_familiarity', 'artist_hotttnesss', 'song_hotttnesss',
    'loudness_max_mean', 'loudness_max_std', 'loudness_start_mean', 'loudness_start_std',
    'num_bars', 'num_beats', 'num_sections', 'num_tatums',
    'bars_confidence_mean', 'beats_confidence_mean', 'sections_confidence_mean', 'tatums_confidence_mean',
    'tempo_estimate', 'num_mbtags', 'num_similar_artists', 'segments_confidence_mean', 'segments_confidence_std',
    'loudness_max_max', 'loudness_max_min'
]

In [28]:

# Extract timbre and pitches as flattened arrays (add to features)
timbre_features = np.array([arr.flatten() for arr in df['timbre_mean'].values])  # 12 dims
pitches_features = np.array([arr.flatten() for arr in df['pitches_mean'].values])  # 12 dims

# Combine all features
numeric_data = df[numeric_features].fillna(0).values
X = np.hstack([numeric_data, timbre_features, pitches_features])

print(f"Feature matrix shape before reduction: {X.shape}")

Feature matrix shape before reduction: (10000, 46)


In [29]:

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Compute similarity on original scaled features
similarity_scores = cosine_similarity(X_scaled)
print(f"Original similarity matrix shape: {similarity_scores.shape}")

Original similarity matrix shape: (10000, 10000)


In [30]:
# Recommendation function
def get_recommendations(song_title=None, song_idx=None, top_n=10):
    """Get top N song recommendations"""
    
    # Get song index by title or use provided index
    if song_title:
        mask = df['title'].str.contains(song_title, case=False, na=False)
        matches = df[mask]
        if len(matches) == 0:
            print(f"Song '{song_title}' not found")
            return
        song_idx = matches.index[0]
    
    if song_idx is None:
        print("Provide either song_title or song_idx")
        return
    
    # Validate index
    if song_idx >= len(df) or song_idx < 0:
        print(f"Invalid song index: {song_idx}")
        return
    
    # Get similarity scores
    scores = similarity_scores[song_idx]
    # Get top N most similar (excluding the song itself)
    similar_indices = np.argsort(scores)[::-1][1:top_n+1]
    
    print(f"Recommendations similar to:")
    print(f"'{df.iloc[song_idx]['title']}' by {df.iloc[song_idx]['artist']}")
    
    for rank, idx in enumerate(similar_indices, 1):
        sim_score = scores[idx]
        print(f"{rank}. '{df.iloc[idx]['title']}' by {df.iloc[idx]['artist']}")
        print(f"   Release: {df.iloc[idx]['release']} | Similarity: {sim_score:.3f}")

# Test recommendation
get_recommendations(song_idx=0, top_n=10)


Recommendations similar to:
'I Didn't Mean To' by Casual
1. 'Check It (Explicit)' by Lords Of The Underground
   Release: Here Come the Lords | Similarity: 0.797
2. 'Only When I'm Drunk' by Tha Alkaholiks
   Release: 21 & Over | Similarity: 0.717
3. 'Heavenly Father' by FU-Schnickens
   Release: Greatest Hits | Similarity: 0.708
4. 'Hippa To Da Hoppa [Explicit Version]' by Ol' Dirty Bastard
   Release: Return To The 36 Chambers: The Dirty Version | Similarity: 0.693
5. 'Get-U-Now' by KMD
   Release: Bl_ck B_st_rds | Similarity: 0.687
6. 'True Fuschnick' by FU-Schnickens
   Release: Greatest Hits | Similarity: 0.667
7. 'A B***** Is A B***** (Edited)' by N.W.A.
   Release: Straight Outta Compton | Similarity: 0.657
8. 'Jane 5' by EPMD
   Release: Back In Business | Similarity: 0.643
9. 'Ho Shit (Chopped&Screwed)' by 5th Ward Boyz
   Release: Ghetto Dope | Similarity: 0.632
10. 'Cheddar Chasin'' by Lunasicc
   Release: A Million Words A Million Dollars | Similarity: 0.629


In [31]:
# Save the recommendation model
import pickle

model_data = {
    'similarity_scores': similarity_scores,
    'scaler': scaler,
    'feature_list': numeric_features,
    'df_metadata': df[['song_id', 'title', 'artist', 'release']].reset_index(drop=True)
}

model_path = 'processed_data/recommendation_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"Model saved: {model_path}")

Model saved: processed_data/recommendation_model.pkl


In [1]:
import pickle
import numpy as np

# Load the saved model
with open('processed_data/recommendation_model.pkl', 'rb') as f:
    model = pickle.load(f)

similarity_scores = model['similarity_scores']
df_metadata = model['df_metadata']

# Use for recommendations
def get_recommendations_from_saved(song_idx, top_n=10):
    result=[]
    scores = similarity_scores[song_idx]
    similar_indices = np.argsort(scores)[::-1][1:top_n+1]
    
    for rank, idx in enumerate(similar_indices, 1):
        result.append((rank, df_metadata.iloc[idx]['title'], df_metadata.iloc[idx]['artist']))
    return result

result = get_recommendations_from_saved(0, top_n=5)

print(result)

[(1, 'Check It (Explicit)', 'Lords Of The Underground'), (2, "Only When I'm Drunk", 'Tha Alkaholiks'), (3, 'Heavenly Father', 'FU-Schnickens'), (4, 'Hippa To Da Hoppa [Explicit Version]', "Ol' Dirty Bastard"), (5, 'Get-U-Now', 'KMD')]
